# NRC-VAD Valence Sentiment

This notebook estimates annual connotational sentiment for ADHD, Autism, and the three baseline terms. It keeps the Baes-style local-collocate index already implemented here, but now joins the locked frame-classifier output so target estimates are substantive-frame aware.


## Setup

The diachronic axis is publication year (`lsc_year`). ADHD and Autism target contexts are restricted to substantive core discourse for semantic estimates: `clinical_only`, `lived_only`, and `mixed`. Baselines are not frame-labelled and remain separate comparator series.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import hashlib
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import spacy

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
NRC_VAD_PATH = PROJECT_ROOT / "data/external/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt"
INTERIM_VAD_DIR = PROJECT_ROOT / "data/interim/lsc/vad"
SENTIMENT_DIR = PROJECT_ROOT / "data/processed/lsc/sentiment"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/sentiment"

INTERIM_VAD_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VAD_MATCHES_PATH = INTERIM_VAD_DIR / "lsc_vad_collocate_matches.parquet"
VAD_CONTEXT_COVERAGE_PATH = INTERIM_VAD_DIR / "lsc_vad_context_coverage.parquet"
ANNUAL_VALENCE_PATH = SENTIMENT_DIR / "lsc_sentiment_annual_valence.csv"
COVERAGE_PATH = SENTIMENT_DIR / "lsc_sentiment_coverage.csv"
TOP_COLLOCATES_PATH = SENTIMENT_DIR / "lsc_sentiment_top_collocates.csv"
AUDIT_FLAGS_PATH = SENTIMENT_DIR / "lsc_sentiment_audit_flags.csv"
TREND_SUMMARY_PATH = SENTIMENT_DIR / "lsc_sentiment_trend_models.csv"
FRAME_CONTEXT_DIAGNOSTICS_PATH = SENTIMENT_DIR / "lsc_sentiment_frame_context_diagnostics.csv"
VALENCE_PLOT_PATH = FIGURE_DIR / "lsc_sentiment_valence_trajectories.png"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_YEARS = list(range(2014, 2027))
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"
BOOTSTRAP_REPETITIONS = 500
BOOTSTRAP_SEED = 123
LOW_MATCHED_TOKEN_COVERAGE_WARN = 0.35
LOW_CONTEXT_COVERAGE_WARN = 0.50
TOP_COLLOCATE_SHARE_WARN = 0.20
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75

LSC_UNIT_LABELS = {
    "ADHD": "ADHD",
    "Autism": "Autism",
    "frustration": "Frustration",
    "loneliness": "Loneliness",
    "sadness": "Sadness",
}
LSC_UNIT_COLORS = {
    "ADHD": "#2F6F9F",
    "Autism": "#B66A4A",
    "frustration": "#4F8F78",
    "loneliness": "#7FA68A",
    "sadness": "#9AA6A1",
}
LSC_UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
    "mixed": "Mixed clinical/lived framing",
    "unframed_baseline": "Comparator term",
}
FRAME_COLORS = {
    "substantive_core_overall": "#263238",
    "clinical_only": "#4F8DB3",
    "lived_only": "#C98263",
    "mixed": "#79A889",
    "substantive_other": "#A998C9",
    "non_substantive_or_insufficient": "#B8C0C5",
    "unframed_baseline": "#7B8785",
}
CONDITION_FRAME_COLORS = {
    "ADHD": {
        "substantive_core_overall": "#2F6F9F",
        "clinical_only": "#75A9C8",
        "lived_only": "#AECFE0",
        "mixed": "#D4E4EC",
    },
    "Autism": {
        "substantive_core_overall": "#B66A4A",
        "clinical_only": "#CE8D70",
        "lived_only": "#E1B49D",
        "mixed": "#F2D8CF",
    },
}
FRAME_MARKERS = {
    "substantive_core_overall": "o",
    "clinical_only": "s",
    "lived_only": "^",
    "mixed": "D",
    "unframed_baseline": "o",
}
LSC_FIGURE_DPI = 300

assert CONTEXT_PATH.exists(), CONTEXT_PATH
assert FRAME_LABEL_PATH.exists(), FRAME_LABEL_PATH
assert NRC_VAD_PATH.exists(), NRC_VAD_PATH

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise OSError(
        "Missing spaCy model en_core_web_sm. Install it with: python -m spacy download en_core_web_sm"
    ) from exc

BAES_EXCLUDED_POS = {"PUNCT", "SYM", "PART", "SPACE", "NUM"}
BAES_EXTRA_STOPWORDS = {"%", "em", "<", ">", "\u03b2", "\u00e0", "\u00d7", "+", "@"}
COLLOCATE_STOPWORDS = set(nlp.Defaults.stop_words) | BAES_EXTRA_STOPWORDS


def normalised_lemma(token: spacy.tokens.Token) -> str:
    lemma = token.lemma_.lower().strip() if token.lemma_ else token.text.lower().strip()
    if lemma == "-pron-":
        lemma = token.text.lower().strip()
    return lemma


def is_baes_collocate_token(token: spacy.tokens.Token) -> bool:
    lemma = normalised_lemma(token)
    lower = token.text.lower().strip()
    if token.pos_ in BAES_EXCLUDED_POS:
        return False
    if token.is_space or token.is_punct or token.like_num:
        return False
    if not token.is_alpha or len(lemma) <= 1:
        return False
    if lower in COLLOCATE_STOPWORDS or lemma in COLLOCATE_STOPWORDS:
        return False
    return True

PROJECT_ROOT


PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak')

## Load Inputs And Join Frame Labels

The stable `context_id` is reconstructed from the same fields used by the classifier application notebook. Target contexts in the three core substantive frames are duplicated into their hard frame stratum and the `substantive_core_overall` aggregate. Non-substantive and substantive-other target contexts are retained only in the diagnostics table.


In [2]:
context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "collapsed_raw_forms",
    "collapsed_matched_texts",
    "registered_domain",
    "token_window_5",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["source_context_row_id"] = contexts.index.astype(int)

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")
if contexts["token_window_5"].fillna("").str.strip().eq("").any():
    raise RuntimeError("Some context rows have empty token_window_5 values.")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


label_columns = [
    "context_id",
    "predicted_derived_frame",
    "p_substantive",
    "p_clinical_given_substantive",
    "p_lived_given_substantive",
]
frame_labels = pd.read_csv(FRAME_LABEL_PATH, usecols=label_columns)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context_id values.")

target_mask = contexts["analysis_unit"].isin(TARGET_UNITS)
contexts.loc[target_mask, "context_id"] = contexts.loc[target_mask].apply(stable_context_id, axis=1)
contexts = contexts.merge(frame_labels, on="context_id", how="left")
missing_target_labels = contexts.loc[target_mask, "predicted_derived_frame"].isna().sum()
if missing_target_labels:
    raise RuntimeError(f"Missing frame labels for {missing_target_labels:,} target contexts.")

frame_context_diagnostics = (
    contexts.loc[target_mask]
    .groupby(["analysis_unit", "lsc_year", "predicted_derived_frame"], as_index=False)
    .agg(contexts=("source_context_row_id", "size"), documents=("doc_id", "nunique"))
)
frame_context_diagnostics["included_in_semantic_estimates"] = frame_context_diagnostics[
    "predicted_derived_frame"
].isin(CORE_TARGET_FRAMES)
frame_context_diagnostics["small_cell_flag"] = frame_context_diagnostics[
    "included_in_semantic_estimates"
] & (
    frame_context_diagnostics["contexts"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | frame_context_diagnostics["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)
frame_context_diagnostics.to_csv(FRAME_CONTEXT_DIAGNOSTICS_PATH, index=False)

baseline_contexts = contexts.loc[~target_mask].copy()
baseline_contexts["frame_stratum"] = BASELINE_FRAME_STRATUM

core_target_contexts = contexts.loc[
    target_mask & contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)
].copy()
target_by_frame = core_target_contexts.copy()
target_by_frame["frame_stratum"] = target_by_frame["predicted_derived_frame"]
target_overall = core_target_contexts.copy()
target_overall["frame_stratum"] = "substantive_core_overall"

analysis_contexts = pd.concat([baseline_contexts, target_overall, target_by_frame], ignore_index=True, sort=False)
analysis_contexts["context_row_id"] = np.arange(len(analysis_contexts), dtype=int)

vad = pd.read_csv(NRC_VAD_PATH, sep="\t")
expected_vad_columns = {"term", "valence", "arousal", "dominance"}
if set(vad.columns) != expected_vad_columns:
    raise RuntimeError(f"Unexpected NRC-VAD columns: {vad.columns.tolist()}")
for column in ["valence", "arousal", "dominance"]:
    if not vad[column].between(-1, 1).all():
        raise RuntimeError(f"NRC-VAD {column} values outside expected [-1, 1] range.")

analysis_context_summary = (
    analysis_contexts.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(contexts=("context_row_id", "size"), documents=("doc_id", "nunique"))
    .sort_values(["analysis_unit", "frame_stratum"])
)
print(f"Original contexts: {len(contexts):,}")
print(f"Analysis context rows after frame expansion: {len(analysis_contexts):,}")
print(f"NRC-VAD terms: {len(vad):,}")
analysis_context_summary


Original contexts: 293,670
Analysis context rows after frame expansion: 311,030
NRC-VAD terms: 54,801


,analysis_unit,frame_stratum,contexts,documents
0,ADHD,clinical_only,12039,8006
1,ADHD,lived_only,3610,2730
2,ADHD,mixed,2000,1634
3,ADHD,substantive_core_overall,17649,11444
4,Autism,clinical_only,20306,12959
5,Autism,lived_only,12826,9068
6,Autism,mixed,6331,4735
7,Autism,substantive_core_overall,39463,24141
8,frustration,unframed_baseline,105482,93663
9,loneliness,unframed_baseline,38052,30431


## Prepare NRC-VAD Lookup

NRC-VAD v2.1 includes unigrams and multi-word expressions. To use lemmatised context windows consistently, lexicon terms are also tokenised and lemmatised with the same spaCy model. If multiple surface entries collapse to the same lemma phrase, their VAD scores are averaged.


In [3]:
lexicon_records: list[dict[str, object]] = []
dropped_lexicon_entries = 0
vad_terms = vad["term"].fillna("").map(str).tolist()
for doc, row in zip(nlp.pipe(vad_terms, batch_size=1000), vad.itertuples(index=False)):
    if any(not is_baes_collocate_token(token) for token in doc):
        dropped_lexicon_entries += 1
        continue
    lemma_tokens = [normalised_lemma(token) for token in doc]
    if not lemma_tokens:
        dropped_lexicon_entries += 1
        continue
    lexicon_records.append(
        {
            "lexicon_key": " ".join(lemma_tokens),
            "lexicon_tuple": tuple(lemma_tokens),
            "lexicon_length": len(lemma_tokens),
            "term": row.term,
            "valence": float(row.valence),
            "arousal": float(row.arousal),
            "dominance": float(row.dominance),
        }
    )

lexicon = pd.DataFrame(lexicon_records)
lexicon_lookup = (
    lexicon.groupby(["lexicon_key", "lexicon_length"], as_index=False)
    .agg(
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        source_terms=("term", lambda values: " | ".join(sorted(set(values))[:5])),
        source_term_count=("term", "nunique"),
    )
)
lexicon_lookup["lexicon_tuple"] = lexicon_lookup["lexicon_key"].str.split().map(tuple)

unigram_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] == 1].itertuples(index=False)
}
mwe_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] > 1].itertuples(index=False)
}
mwe_lengths = sorted({len(key) for key in mwe_lookup}, reverse=True)

print(f"Normalised NRC-VAD keys: {len(lexicon_lookup):,}")
print(f"Dropped NRC-VAD entries during Baes-style normalisation: {dropped_lexicon_entries:,}")
print(f"Unigram keys: {len(unigram_lookup):,}")
print(f"MWE keys: {len(mwe_lookup):,}")
lexicon_lookup.head()


Normalised NRC-VAD keys: 46,556
Dropped NRC-VAD entries during Baes-style normalisation: 4,633
Unigram keys: 40,809
MWE keys: 5,747


,lexicon_key,lexicon_length,valence,arousal,dominance,source_terms,source_term_count,lexicon_tuple
0,aaaaaaah,1,-0.042,0.212,-0.418,aaaaaaah,1,"(aaaaaaah,)"
1,aaaah,1,0.040,0.272,-0.436,aaaah,1,"(aaaah,)"
2,aardvark,1,-0.146,-0.020,-0.126,aardvark,1,"(aardvark,)"
3,aback,1,-0.230,-0.186,-0.424,aback,1,"(aback,)"
4,abacus,1,0.020,-0.448,-0.030,abacus,1,"(abacus,)"


## Extract VAD-Matched Collocates

The collocate extraction follows the Baes-style local-window procedure: remove the focal mention's lexical material, skip punctuation/symbol/space/number/particle tokens, remove stopwords, and greedily match NRC-VAD multi-word expressions before unigrams. NRC-VAD phrase entries that require stopwords are dropped rather than collapsed into content-only remnants. The saved VAD handoff includes `frame_stratum` so the Intensity notebook can reuse exactly the same frame-aware preprocessing.


In [4]:
TARGET_RAW_FORM_EXCLUSION_TOKENS = {
    "adhd": {"adhd"},
    "attention_deficit": {"attention", "deficit", "hyperactivity", "disorder"},
    "autism": {"autism"},
    "autistic": {"autistic"},
    "autism_spectrum": {"autism", "spectrum"},
    "asd_disambiguated": {"asd"},
}
BASELINE_RAW_FORM_EXCLUSION_TOKENS = {
    "frustration": {"frustration"},
    "sadness": {"sadness"},
    "loneliness": {"loneliness"},
}
RAW_FORM_EXCLUSION_TOKENS_BY_ROLE = {
    "target": TARGET_RAW_FORM_EXCLUSION_TOKENS,
    "baseline": BASELINE_RAW_FORM_EXCLUSION_TOKENS,
}


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def exclusion_tokens_for_row(row: pd.Series) -> set[str]:
    raw_forms = {str(row["raw_form"])} | set(split_pipe_values(row.get("collapsed_raw_forms")))
    role_exclusions = RAW_FORM_EXCLUSION_TOKENS_BY_ROLE.get(str(row["term_role"]), {})
    tokens: set[str] = set()
    for raw_form in raw_forms:
        tokens.update(role_exclusions.get(raw_form, {raw_form.replace("_", " ")}))
    matched_text = str(row.get("matched_text") or "")
    if matched_text:
        tokens.update(part.lower() for part in re.findall(r"[A-Za-z]+", matched_text))
    return tokens


def lexical_tokens(doc, excluded_terms: set[str]) -> list[dict[str, object]]:
    tokens = []
    for token in doc:
        lemma = normalised_lemma(token)
        lower = token.text.lower()
        if not is_baes_collocate_token(token):
            continue
        if lemma in excluded_terms or lower in excluded_terms:
            continue
        tokens.append({"token_index": token.i, "text": token.text, "lemma": lemma})
    return tokens


def match_vad_units(tokens: list[dict[str, object]]) -> list[dict[str, object]]:
    matches: list[dict[str, object]] = []
    index = 0
    while index < len(tokens):
        matched = None
        for length in mwe_lengths:
            if index + length > len(tokens):
                continue
            key = tuple(token["lemma"] for token in tokens[index : index + length])
            if key in mwe_lookup:
                matched = (length, mwe_lookup[key], "mwe")
                break
        if matched is None:
            key = (tokens[index]["lemma"],)
            if key in unigram_lookup:
                matched = (1, unigram_lookup[key], "unigram")
        if matched is None:
            index += 1
            continue
        length, lexicon_row, collocate_type = matched
        span_tokens = tokens[index : index + length]
        matches.append(
            {
                "collocate": lexicon_row.lexicon_key,
                "collocate_type": collocate_type,
                "surface_text": " ".join(str(token["text"]) for token in span_tokens),
                "token_start_in_window": int(span_tokens[0]["token_index"]),
                "token_end_in_window": int(span_tokens[-1]["token_index"] + 1),
                "matched_token_count": int(length),
                "valence": float(lexicon_row.valence),
                "arousal": float(lexicon_row.arousal),
                "dominance": float(lexicon_row.dominance),
                "source_terms": lexicon_row.source_terms,
                "source_term_count": int(lexicon_row.source_term_count),
            }
        )
        index += length
    return matches


match_rows: list[dict[str, object]] = []
coverage_rows: list[dict[str, object]] = []

for doc, (_, row) in zip(
    nlp.pipe(analysis_contexts["token_window_5"].astype(str), batch_size=1000),
    analysis_contexts.iterrows(),
):
    excluded_terms = exclusion_tokens_for_row(row)
    tokens = lexical_tokens(doc, excluded_terms)
    row_matches = match_vad_units(tokens)
    matched_token_positions = sum(match["matched_token_count"] for match in row_matches)

    coverage_rows.append(
        {
            "context_row_id": int(row["context_row_id"]),
            "source_context_row_id": int(row["source_context_row_id"]),
            "context_id": row.get("context_id"),
            "doc_id": row["doc_id"],
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "term_role": row["term_role"],
            "target_group": row["target_group"],
            "raw_form": row["raw_form"],
            "frame_stratum": row["frame_stratum"],
            "predicted_derived_frame": row.get("predicted_derived_frame"),
            "candidate_collocate_tokens": len(tokens),
            "matched_vad_units": len(row_matches),
            "matched_token_positions": int(matched_token_positions),
            "has_vad_match": bool(row_matches),
        }
    )

    for match in row_matches:
        match_rows.append(
            {
                "context_row_id": int(row["context_row_id"]),
                "source_context_row_id": int(row["source_context_row_id"]),
                "context_id": row.get("context_id"),
                "doc_id": row["doc_id"],
                "lsc_year": int(row["lsc_year"]),
                "published_year": int(row["published_year"]),
                "source_year": int(row["source_year"]),
                "analysis_unit": row["analysis_unit"],
                "term_role": row["term_role"],
                "target_group": row["target_group"],
                "raw_form": row["raw_form"],
                "frame_stratum": row["frame_stratum"],
                "predicted_derived_frame": row.get("predicted_derived_frame"),
                "registered_domain": row["registered_domain"],
                **match,
            }
        )

vad_matches = pd.DataFrame(match_rows)
context_coverage = pd.DataFrame(coverage_rows)
if vad_matches.empty:
    raise RuntimeError("No NRC-VAD collocates matched the frame-aware LSC contexts.")
stopword_collocate_rows = vad_matches["collocate"].map(lambda value: any(part in COLLOCATE_STOPWORDS for part in str(value).split())).sum()
if stopword_collocate_rows:
    raise RuntimeError(f"Stopword collocates survived Baes-style filtering: {int(stopword_collocate_rows):,}")

vad_matches.to_parquet(VAD_MATCHES_PATH, index=False)
context_coverage.to_parquet(VAD_CONTEXT_COVERAGE_PATH, index=False)

print(f"VAD match rows: {len(vad_matches):,}")
print(f"Contexts with any VAD match: {context_coverage['has_vad_match'].sum():,} / {len(context_coverage):,}")
vad_matches.head()


VAD match rows: 1,069,580
Contexts with any VAD match: 308,478 / 311,030


,context_row_id,source_context_row_id,context_id,doc_id,lsc_year,published_year,source_year,analysis_unit,term_role,target_group,raw_form,frame_stratum,predicted_derived_frame,registered_domain,collocate,collocate_type,surface_text,token_start_in_window,token_end_in_window,matched_token_count,valence,arousal,dominance,source_terms,source_term_count
0,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,continue,unigram,continued,2,3,1,0.288,-0.088667,0.088,continue | continued | continuing,3
1,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,failure,unigram,failure,3,4,1,-0.666,0.150000,-0.722,failure,1
2,1,11402,NaN,000750cc054d9ff4,2014,2014,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,phassociation.org,husband,unigram,husband,2,3,1,0.542,0.000000,0.310,husband,1
3,1,11402,NaN,000750cc054d9ff4,2014,2014,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,phassociation.org,moment,unigram,moment,8,9,1,0.208,-0.130000,0.018,moment,1
4,2,11403,NaN,000def93004a1f28,2014,2014,2018,frustration,baseline,baseline,frustration,unframed_baseline,NaN,hotnewhiphop.com,recently,unigram,recently,3,4,1,0.000,0.000000,0.000,recently,1


## Annual Valence Index

Annual valence is the weighted mean of matched NRC-VAD collocate occurrences for each analysis unit, publication year, and frame stratum. Coverage is reported alongside the index so sparse or unstable strata are not over-interpreted.


In [5]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_valence = (
    vad_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        valence_mean=("valence", "mean"),
        valence_sd=("valence", "std"),
        arousal_mean_for_reuse=("arousal", "mean"),
        dominance_mean_for_reuse=("dominance", "mean"),
        matched_vad_units=("valence", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_valence = annual_valence.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_valence = annual_valence.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,0.061856,0.488648,0.028210,0.041583,4317,1157,791,1286,808,4888,4317,4443,1249,0.908961,0.971229,False
1,2015,ADHD,clinical_only,target,ADHD,0.074820,0.488885,0.020463,0.045729,3869,1119,763,1203,774,4464,3869,3979,1171,0.891353,0.973400,False
2,2016,ADHD,clinical_only,target,ADHD,0.038802,0.505949,0.031395,0.028800,3833,1112,784,1149,790,4295,3833,3933,1124,0.915716,0.978242,False
3,2017,ADHD,clinical_only,target,ADHD,0.039036,0.491473,0.036331,0.035293,3594,1107,735,1099,751,4123,3594,3709,1063,0.899588,0.967243,False
4,2018,ADHD,clinical_only,target,ADHD,0.039881,0.500777,0.034197,0.024145,4089,1154,782,1182,787,4602,4089,4193,1164,0.911126,0.984772,False
5,2019,ADHD,clinical_only,target,ADHD,0.010419,0.504340,0.038305,0.015428,3475,1004,664,1000,674,3957,3475,3571,981,0.902451,0.981000,False
6,2020,ADHD,clinical_only,target,ADHD,0.031440,0.494384,0.040228,0.025447,3136,949,662,959,668,3590,3136,3237,932,0.901671,0.971846,False
7,2021,ADHD,clinical_only,target,ADHD,0.025841,0.506626,0.034450,0.010944,3177,1033,634,928,636,3565,3177,3272,911,0.917812,0.981681,False
8,2022,ADHD,clinical_only,target,ADHD,0.059404,0.491737,0.015353,0.042366,3202,992,624,930,634,3606,3202,3305,906,0.916528,0.974194,False
9,2023,ADHD,clinical_only,target,ADHD,0.058146,0.486757,0.039470,0.056264,2548,861,470,750,477,2845,2548,2598,729,0.913181,0.972000,False


## Bootstrap Confidence Intervals

Uncertainty is estimated by resampling documents within each analysis-unit/year/frame stratum. This preserves the existing document-level bootstrap contract while making the frame-specific estimates inspectable.


In [6]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows: list[dict[str, object]] = []

doc_scores = (
    vad_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(valence_sum=("valence", "sum"), matched_vad_units=("valence", "size"))
)

for group_values, doc_frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    valence_sums = doc_frame["valence_sum"].to_numpy(dtype=float)
    unit_counts = doc_frame["matched_vad_units"].to_numpy(dtype=float)
    n_docs = len(doc_frame)
    estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        estimates[repetition] = valence_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_rows.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "valence_bootstrap_mean": float(np.nanmean(estimates)),
            "valence_ci_low": float(np.nanpercentile(estimates, 2.5)),
            "valence_ci_high": float(np.nanpercentile(estimates, 97.5)),
        }
    )

bootstrap = pd.DataFrame(bootstrap_rows)
annual_valence = annual_valence.merge(bootstrap, on=GROUP_COLUMNS, how="left")
annual_valence.to_csv(ANNUAL_VALENCE_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,valence_bootstrap_mean,valence_ci_low,valence_ci_high
0,2014,ADHD,clinical_only,target,ADHD,0.061856,0.488648,0.028210,0.041583,4317,1157,791,1286,808,4888,4317,4443,1249,0.908961,0.971229,False,500,doc_id,0.061976,0.043454,0.080461
1,2015,ADHD,clinical_only,target,ADHD,0.074820,0.488885,0.020463,0.045729,3869,1119,763,1203,774,4464,3869,3979,1171,0.891353,0.973400,False,500,doc_id,0.075182,0.055822,0.093834
2,2016,ADHD,clinical_only,target,ADHD,0.038802,0.505949,0.031395,0.028800,3833,1112,784,1149,790,4295,3833,3933,1124,0.915716,0.978242,False,500,doc_id,0.039297,0.017360,0.059906
3,2017,ADHD,clinical_only,target,ADHD,0.039036,0.491473,0.036331,0.035293,3594,1107,735,1099,751,4123,3594,3709,1063,0.899588,0.967243,False,500,doc_id,0.039716,0.019686,0.058888
4,2018,ADHD,clinical_only,target,ADHD,0.039881,0.500777,0.034197,0.024145,4089,1154,782,1182,787,4602,4089,4193,1164,0.911126,0.984772,False,500,doc_id,0.039649,0.021102,0.058522
5,2019,ADHD,clinical_only,target,ADHD,0.010419,0.504340,0.038305,0.015428,3475,1004,664,1000,674,3957,3475,3571,981,0.902451,0.981000,False,500,doc_id,0.009895,-0.011601,0.033736
6,2020,ADHD,clinical_only,target,ADHD,0.031440,0.494384,0.040228,0.025447,3136,949,662,959,668,3590,3136,3237,932,0.901671,0.971846,False,500,doc_id,0.031383,0.007516,0.054544
7,2021,ADHD,clinical_only,target,ADHD,0.025841,0.506626,0.034450,0.010944,3177,1033,634,928,636,3565,3177,3272,911,0.917812,0.981681,False,500,doc_id,0.025037,0.001189,0.046816
8,2022,ADHD,clinical_only,target,ADHD,0.059404,0.491737,0.015353,0.042366,3202,992,624,930,634,3606,3202,3305,906,0.916528,0.974194,False,500,doc_id,0.058686,0.037750,0.079390
9,2023,ADHD,clinical_only,target,ADHD,0.058146,0.486757,0.039470,0.056264,2548,861,470,750,477,2845,2548,2598,729,0.913181,0.972000,False,500,doc_id,0.058286,0.033427,0.082517


## Trend Models

Following Baes et al.'s analytical strategy, each reported annual series gets a compact trend model. OLS on centred year is the main descriptive summary; an AR(1)-transformed sensitivity slope is reported only when the residual Durbin-Watson diagnostic is flagged. Quadratic fit is retained as a diagnostic rather than a default model.


In [7]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_valence.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "valence_mean",
            **fit_trend(frame, "valence_mean"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)
trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,valence_mean,13,2020.0,0.049085,0.001251,0.001528,0.430318,0.057439,-0.028249,0.239663,1.011305,0.450642,True,0.003551,0.217587,0.575050,0.603299
1,ADHD,lived_only,target,ADHD,valence_mean,13,2020.0,0.149418,0.000178,0.001705,0.918687,0.000991,-0.089828,0.031479,2.389194,-0.559946,False,NaN,NaN,-0.133875,-0.044047
2,ADHD,mixed,target,ADHD,valence_mean,13,2020.0,0.109770,0.004979,0.002513,0.073066,0.263072,0.196079,0.512906,1.653253,0.055126,False,NaN,NaN,0.342995,0.146916
3,ADHD,substantive_core_overall,target,ADHD,valence_mean,13,2020.0,0.075915,0.002330,0.001353,0.113000,0.212366,0.140762,0.460831,1.090441,0.432946,True,0.003749,0.147307,0.430041,0.289279
4,Autism,clinical_only,target,Autism,valence_mean,13,2020.0,0.085089,0.002250,0.001470,0.154234,0.175482,0.100525,0.418905,0.983265,0.404282,True,0.005182,0.039695,0.746846,0.646320
5,Autism,lived_only,target,Autism,valence_mean,13,2020.0,0.230395,-0.003829,0.002230,0.113910,0.211420,0.139730,-0.459804,2.297452,-0.193422,False,NaN,NaN,0.180621,0.040890
6,Autism,mixed,target,Autism,valence_mean,13,2020.0,0.205801,0.001425,0.000951,0.162171,0.169508,0.094009,0.411713,1.792058,0.084900,False,NaN,NaN,0.206762,0.112753
7,Autism,substantive_core_overall,target,Autism,valence_mean,13,2020.0,0.149028,0.001166,0.000606,0.080539,0.251884,0.183874,0.501881,2.280621,-0.523190,False,NaN,NaN,0.282119,0.098245
8,frustration,unframed_baseline,baseline,baseline,valence_mean,13,2020.0,0.092552,0.003729,0.001255,0.012733,0.445117,0.394673,0.667171,0.389310,0.761637,True,0.011628,0.002796,0.892440,0.497767
9,loneliness,unframed_baseline,baseline,baseline,valence_mean,13,2020.0,0.033603,0.004444,0.001343,0.006968,0.498820,0.453258,0.706272,0.892402,0.392622,True,0.006977,0.008906,0.832307,0.379049


## Contributor Diagnostics

The top-collocate table identifies which NRC-VAD matches contribute most to positive or negative annual scores within each frame stratum. These diagnostics guide interpretation and help detect cases where a few words dominate a trajectory.


In [8]:
collocate_counts = (
    vad_matches.groupby(["analysis_unit", "frame_stratum", "lsc_year", "collocate", "collocate_type"], as_index=False)
    .agg(
        count=("collocate", "size"),
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        documents=("doc_id", "nunique"),
    )
)
collocate_counts["weighted_valence_contribution"] = collocate_counts["count"] * collocate_counts["valence"]
collocate_counts["abs_weighted_valence_contribution"] = collocate_counts["weighted_valence_contribution"].abs()
collocate_counts["total_matches_for_unit_year_frame"] = collocate_counts.groupby(
    ["analysis_unit", "frame_stratum", "lsc_year"]
)["count"].transform("sum")
collocate_counts["match_share"] = collocate_counts["count"] / collocate_counts["total_matches_for_unit_year_frame"]

positive_top = (
    collocate_counts.sort_values(
        ["analysis_unit", "frame_stratum", "lsc_year", "weighted_valence_contribution"],
        ascending=[True, True, True, False],
    )
    .groupby(["analysis_unit", "frame_stratum", "lsc_year"])
    .head(10)
    .assign(contribution_direction="positive")
)
negative_top = (
    collocate_counts.sort_values(
        ["analysis_unit", "frame_stratum", "lsc_year", "weighted_valence_contribution"],
        ascending=[True, True, True, True],
    )
    .groupby(["analysis_unit", "frame_stratum", "lsc_year"])
    .head(10)
    .assign(contribution_direction="negative")
)
top_collocates = pd.concat([positive_top, negative_top], ignore_index=True).sort_values(
    ["analysis_unit", "frame_stratum", "lsc_year", "contribution_direction", "abs_weighted_valence_contribution"],
    ascending=[True, True, True, True, False],
)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)
top_collocates.head(20)


,analysis_unit,frame_stratum,lsc_year,collocate,collocate_type,count,valence,arousal,dominance,documents,weighted_valence_contribution,abs_weighted_valence_contribution,total_matches_for_unit_year_frame,match_share,contribution_direction
1430,ADHD,clinical_only,2014,disorder,unigram,84,-0.675000,0.530000,-0.494000,72,-56.700000,56.700000,4317,0.019458,negative
1431,ADHD,clinical_only,2014,depression,unigram,48,-0.938000,0.040000,-0.536000,48,-45.024000,45.024000,4317,0.011119,negative
1432,ADHD,clinical_only,2014,autism,unigram,80,-0.530000,0.000000,-0.106000,73,-42.400000,42.400000,4317,0.018531,negative
1433,ADHD,clinical_only,2014,diagnose,unigram,78,-0.455500,-0.035500,0.019500,68,-35.529000,35.529000,4317,0.018068,negative
1434,ADHD,clinical_only,2014,problem,unigram,37,-0.876000,0.288000,-0.170000,35,-32.412000,32.412000,4317,0.008571,negative
1435,ADHD,clinical_only,2014,drug,unigram,48,-0.665667,0.542000,-0.425000,43,-31.952000,31.952000,4317,0.011119,negative
1436,ADHD,clinical_only,2014,anxiety,unigram,29,-0.708000,0.730000,-0.316000,29,-20.532000,20.532000,4317,0.006718,negative
1437,ADHD,clinical_only,2014,suffer,unigram,21,-0.898000,0.629000,-0.506000,21,-18.858000,18.858000,4317,0.004864,negative
1438,ADHD,clinical_only,2014,hyperactivity,unigram,26,-0.667000,1.000000,-0.333000,23,-17.342000,17.342000,4317,0.006023,negative
1439,ADHD,clinical_only,2014,disability,unigram,18,-0.854000,-0.036000,-0.654000,17,-15.372000,15.372000,4317,0.004170,negative


## Plots And Audit Flags

The report-facing trajectory figure uses three equal-width panels: ADHD, Autism, and comparator terms. The ADHD and Autism panels foreground the substantive-core Overall trajectory and add clinical/disorder and lived-experience traces as lighter contextual lines.

Mixed-frame estimates, coverage diagnostics, small-cell warnings, collocate concentration, and trend diagnostics remain available in the saved CSV tables and audit flags. They are not saved as separate report figures in order to keep the figure folder aligned with the main dissertation story.


In [9]:
READER_FRAME_STRATA = ["clinical_only", "lived_only"]
READER_FRAME_LABELS = {
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
}


def save_lsc_figure(fig: plt.Figure, png_path: Path) -> Path:
    fig.tight_layout(pad=1.1, rect=[0, 0, 1, 0.93])
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def series_color(unit: str, frame_stratum: str) -> str:
    if unit in CONDITION_FRAME_COLORS and frame_stratum in CONDITION_FRAME_COLORS[unit]:
        return CONDITION_FRAME_COLORS[unit][frame_stratum]
    if unit in LSC_UNIT_COLORS:
        return LSC_UNIT_COLORS[unit]
    return FRAME_COLORS.get(frame_stratum, "#7B8785")


def trend_line_for(series: pd.DataFrame, trend: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    years = series["lsc_year"].to_numpy(dtype=float)
    fitted = trend["linear_intercept"] + trend["linear_slope_per_year"] * (years - trend["year_center"])
    return years, fitted


def trend_for(unit: str, frame_stratum: str) -> pd.Series | None:
    row = trend_summary.loc[
        trend_summary["analysis_unit"].eq(unit) & trend_summary["frame_stratum"].eq(frame_stratum)
    ]
    if row.empty or pd.isna(row.iloc[0]["linear_slope_per_year"]):
        return None
    return row.iloc[0]


def y_limits_from(frame: pd.DataFrame, value_column: str, ci_low: str | None = None, ci_high: str | None = None) -> tuple[float, float]:
    values = [frame[value_column].to_numpy(dtype=float)]
    if ci_low and ci_low in frame:
        values.append(frame[ci_low].to_numpy(dtype=float))
    if ci_high and ci_high in frame:
        values.append(frame[ci_high].to_numpy(dtype=float))
    finite_values = [v[np.isfinite(v)] for v in values if len(v)]
    combined = np.concatenate(finite_values)
    low, high = float(combined.min()), float(combined.max())
    padding = max((high - low) * 0.10, 0.004)
    return low - padding, high + padding


def style_year_axis(ax: plt.Axes) -> None:
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(axis="x", labelsize=8.5)


def plot_line_with_trend(
    ax: plt.Axes,
    frame: pd.DataFrame,
    unit: str,
    frame_stratum: str,
    value_column: str,
    color: str,
    marker: str,
    label: str | None = None,
    ci_low: str | None = None,
    ci_high: str | None = None,
    ribbon_alpha: float = 0.12,
    linewidth: float = 2.2,
    markersize: float = 4.8,
    alpha: float = 1.0,
    show_trend: bool = True,
) -> None:
    series = frame.loc[frame["analysis_unit"].eq(unit) & frame["frame_stratum"].eq(frame_stratum)].sort_values("lsc_year")
    if series.empty:
        return
    ax.plot(
        series["lsc_year"],
        series[value_column],
        marker=marker,
        markersize=markersize,
        linewidth=linewidth,
        label=label,
        color=color,
        alpha=alpha,
    )
    if ci_low and ci_high:
        ax.fill_between(
            series["lsc_year"].to_numpy(dtype=float),
            series[ci_low].to_numpy(dtype=float),
            series[ci_high].to_numpy(dtype=float),
            color=color,
            alpha=ribbon_alpha,
            linewidth=0,
        )
    trend = trend_for(unit, frame_stratum)
    if show_trend and trend is not None:
        years, fitted = trend_line_for(series, trend)
        ax.plot(years, fitted, color=color, linewidth=1.05, linestyle="--", alpha=min(alpha + 0.12, 0.92))


def plot_target_panel(ax: plt.Axes, unit: str) -> None:
    plot_line_with_trend(
        ax,
        annual_valence,
        unit,
        "substantive_core_overall",
        "valence_mean",
        series_color(unit, "substantive_core_overall"),
        FRAME_MARKERS["substantive_core_overall"],
        "Overall",
        "valence_ci_low",
        "valence_ci_high",
        ribbon_alpha=0.14,
        linewidth=2.8,
        markersize=4.8,
    )
    for frame_stratum in READER_FRAME_STRATA:
        plot_line_with_trend(
            ax,
            annual_valence,
            unit,
            frame_stratum,
            "valence_mean",
            series_color(unit, frame_stratum),
            FRAME_MARKERS[frame_stratum],
            READER_FRAME_LABELS[frame_stratum],
            "valence_ci_low",
            "valence_ci_high",
            ribbon_alpha=0.075,
            linewidth=1.55,
            markersize=3.7,
            alpha=0.82,
        )
    ax.set_title(unit, loc="left", fontsize=12, fontweight="bold")
    style_year_axis(ax)
    ax.legend(loc="best", fontsize=7.6)


def plot_baseline_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    value_column: str,
    ci_low: str | None = None,
    ci_high: str | None = None,
    show_trend: bool = True,
) -> None:
    for unit in BASELINE_UNITS:
        plot_line_with_trend(
            ax,
            frame,
            unit,
            BASELINE_FRAME_STRATUM,
            value_column,
            LSC_UNIT_COLORS[unit],
            UNIT_MARKERS[unit] if "UNIT_MARKERS" in globals() else LSC_UNIT_MARKERS[unit],
            LSC_UNIT_LABELS[unit] if "LSC_UNIT_LABELS" in globals() else unit,
            ci_low,
            ci_high,
            ribbon_alpha=0.08,
            linewidth=2.0,
            show_trend=show_trend,
        )
    ax.set_title("Comparator terms", loc="left", fontsize=12, fontweight="bold")
    ax.legend(loc="best", fontsize=7.6)
    style_year_axis(ax)


main_rows = pd.concat(
    [
        annual_valence.loc[
            annual_valence["analysis_unit"].isin(TARGET_UNITS)
            & annual_valence["frame_stratum"].isin(["substantive_core_overall", *READER_FRAME_STRATA])
        ],
        annual_valence.loc[annual_valence["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    ],
    ignore_index=True,
)
main_ylim = y_limits_from(main_rows, "valence_mean", "valence_ci_low", "valence_ci_high")

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.35), sharex=True, sharey=True)
fig.suptitle("Sentiment: valence near target terms", fontsize=15, fontweight="bold", x=0.02, ha="left")
for ax, unit in zip(axes[:2], TARGET_UNITS):
    plot_target_panel(ax, unit)
plot_baseline_panel(
    axes[2],
    annual_valence.loc[annual_valence["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    "valence_mean",
    "valence_ci_low",
    "valence_ci_high",
)
for ax in axes:
    ax.set_ylim(*main_ylim)
    ax.set_xlabel("Publication year")
axes[0].set_ylabel("Mean valence (-1 to 1)")
valence_pdf_path = save_lsc_figure(fig, VALENCE_PLOT_PATH)
plt.close(fig)

flags: list[dict[str, object]] = []
for _, row in coverage.iterrows():
    if row["matched_token_coverage"] < LOW_MATCHED_TOKEN_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_matched_token_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["matched_token_coverage"]),
            }
        )
    if row["context_match_coverage"] < LOW_CONTEXT_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_context_match_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["context_match_coverage"]),
            }
        )
    if bool(row.get("small_cell_flag", False)):
        flags.append(
            {
                "severity": "note",
                "category": "small_frame_year_cell",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["context_rows"]),
                "detail": f"documents={int(row['documents'])}",
            }
        )

for _, row in collocate_counts.loc[collocate_counts["match_share"] >= TOP_COLLOCATE_SHARE_WARN].iterrows():
    flags.append(
        {
            "severity": "warn",
            "category": "top_collocate_concentration",
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "frame_stratum": row["frame_stratum"],
            "value": float(row["match_share"]),
            "detail": row["collocate"],
        }
    )

for _, row in trend_summary.loc[trend_summary["autocorrelation_flag"]].iterrows():
    flags.append(
        {
            "severity": "note",
            "category": "trend_residual_autocorrelation",
            "lsc_year": pd.NA,
            "analysis_unit": row["analysis_unit"],
            "frame_stratum": row["frame_stratum"],
            "value": float(row["durbin_watson"]),
            "detail": "AR(1) sensitivity slope reported in trend table",
        }
    )

audit_flags = pd.DataFrame(flags, columns=["severity", "category", "lsc_year", "analysis_unit", "frame_stratum", "value", "detail"])
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

print(f"Wrote {ANNUAL_VALENCE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TOP_COLLOCATES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TREND_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {FRAME_CONTEXT_DIAGNOSTICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_MATCHES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_CONTEXT_COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VALENCE_PLOT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {valence_pdf_path.relative_to(PROJECT_ROOT)}")
print(f"Audit warnings/notes: {len(audit_flags):,}")
annual_valence.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).head(20)


Wrote data/processed/lsc/sentiment/lsc_sentiment_annual_valence.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_coverage.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_top_collocates.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_trend_models.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_frame_context_diagnostics.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_audit_flags.csv
Wrote data/interim/lsc/vad/lsc_vad_collocate_matches.parquet
Wrote data/interim/lsc/vad/lsc_vad_context_coverage.parquet
Wrote reports/figures/lsc/sentiment/lsc_sentiment_valence_trajectories.png
Wrote reports/figures/lsc/sentiment/lsc_sentiment_valence_trajectories.pdf
Audit warnings/notes: 6


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,valence_bootstrap_mean,valence_ci_low,valence_ci_high
0,2014,ADHD,clinical_only,target,ADHD,0.061856,0.488648,0.028210,0.041583,4317,1157,791,1286,808,4888,4317,4443,1249,0.908961,0.971229,False,500,doc_id,0.061976,0.043454,0.080461
1,2015,ADHD,clinical_only,target,ADHD,0.074820,0.488885,0.020463,0.045729,3869,1119,763,1203,774,4464,3869,3979,1171,0.891353,0.973400,False,500,doc_id,0.075182,0.055822,0.093834
2,2016,ADHD,clinical_only,target,ADHD,0.038802,0.505949,0.031395,0.028800,3833,1112,784,1149,790,4295,3833,3933,1124,0.915716,0.978242,False,500,doc_id,0.039297,0.017360,0.059906
3,2017,ADHD,clinical_only,target,ADHD,0.039036,0.491473,0.036331,0.035293,3594,1107,735,1099,751,4123,3594,3709,1063,0.899588,0.967243,False,500,doc_id,0.039716,0.019686,0.058888
4,2018,ADHD,clinical_only,target,ADHD,0.039881,0.500777,0.034197,0.024145,4089,1154,782,1182,787,4602,4089,4193,1164,0.911126,0.984772,False,500,doc_id,0.039649,0.021102,0.058522
5,2019,ADHD,clinical_only,target,ADHD,0.010419,0.504340,0.038305,0.015428,3475,1004,664,1000,674,3957,3475,3571,981,0.902451,0.981000,False,500,doc_id,0.009895,-0.011601,0.033736
6,2020,ADHD,clinical_only,target,ADHD,0.031440,0.494384,0.040228,0.025447,3136,949,662,959,668,3590,3136,3237,932,0.901671,0.971846,False,500,doc_id,0.031383,0.007516,0.054544
7,2021,ADHD,clinical_only,target,ADHD,0.025841,0.506626,0.034450,0.010944,3177,1033,634,928,636,3565,3177,3272,911,0.917812,0.981681,False,500,doc_id,0.025037,0.001189,0.046816
8,2022,ADHD,clinical_only,target,ADHD,0.059404,0.491737,0.015353,0.042366,3202,992,624,930,634,3606,3202,3305,906,0.916528,0.974194,False,500,doc_id,0.058686,0.037750,0.079390
9,2023,ADHD,clinical_only,target,ADHD,0.058146,0.486757,0.039470,0.056264,2548,861,470,750,477,2845,2548,2598,729,0.913181,0.972000,False,500,doc_id,0.058286,0.033427,0.082517


## Compact Handoff Summary

The handoff summary keeps the rerun easy to inspect: it shows the mean, sample standard deviation, and range of the annual valence estimates alongside coverage, warning counts, and trend slopes. The descriptive mean and standard deviation summarise annual trajectory values rather than collocate- or context-level observations.


In [10]:
handoff_summary = (
    annual_valence.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        years=("lsc_year", "nunique"),
        valence_annual_mean=("valence_mean", "mean"),
        valence_annual_sd=("valence_mean", "std"),
        valence_min=("valence_mean", "min"),
        valence_max=("valence_mean", "max"),
        matched_units=("matched_vad_units", "sum"),
        documents_with_matches=("documents_with_matches", "sum"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        min_context_match_coverage=("context_match_coverage", "min"),
        small_cell_years=("small_cell_flag", "sum"),
    )
)
warning_counts = (
    audit_flags.groupby(["analysis_unit", "frame_stratum"]).size().rename("warnings").reset_index()
    if not audit_flags.empty
    else pd.DataFrame({"analysis_unit": [], "frame_stratum": [], "warnings": []})
)
handoff_summary = handoff_summary.merge(warning_counts, on=["analysis_unit", "frame_stratum"], how="left").fillna({"warnings": 0})
handoff_summary = handoff_summary.merge(
    trend_summary[["analysis_unit", "frame_stratum", "linear_slope_per_year", "linear_p_value", "autocorrelation_flag"]],
    on=["analysis_unit", "frame_stratum"],
    how="left",
)
handoff_summary["warnings"] = handoff_summary["warnings"].astype(int)
handoff_summary["small_cell_years"] = handoff_summary["small_cell_years"].astype(int)
handoff_summary


,analysis_unit,frame_stratum,years,valence_annual_mean,valence_annual_sd,valence_min,valence_max,matched_units,documents_with_matches,min_matched_token_coverage,min_context_match_coverage,small_cell_years,warnings,linear_slope_per_year,linear_p_value,autocorrelation_flag
0,ADHD,clinical_only,13,0.049085,0.020329,0.010419,0.075965,40772,7903,0.888416,0.961938,0,1,0.001251,0.430318,True
1,ADHD,lived_only,13,0.149418,0.022039,0.102658,0.188627,11246,2702,0.864426,0.953704,0,0,0.000178,0.918687,False
2,ADHD,mixed,13,0.109770,0.037807,0.045126,0.167899,6648,1609,0.863436,0.966667,1,1,0.004979,0.073066,False
3,ADHD,substantive_core_overall,13,0.075915,0.019690,0.043627,0.105550,58666,11319,0.887415,0.962389,0,1,0.002330,0.113000,True
4,Autism,clinical_only,13,0.085089,0.020917,0.057263,0.127151,78302,12932,0.879640,0.991758,0,1,0.002250,0.154234,True
5,Autism,lived_only,13,0.230395,0.032434,0.126130,0.253100,43105,8995,0.865809,0.986175,0,0,-0.003829,0.113910,False
6,Autism,mixed,13,0.205801,0.013475,0.184616,0.223290,23261,4713,0.832168,0.966667,0,0,0.001425,0.162171,False
7,Autism,substantive_core_overall,13,0.149028,0.009044,0.136792,0.173325,144668,24049,0.876249,0.989362,0,0,0.001166,0.080539,False
8,frustration,unframed_baseline,13,0.092552,0.021768,0.075288,0.146256,360766,93157,0.901552,0.988223,0,1,0.003729,0.012733,True
9,loneliness,unframed_baseline,13,0.033603,0.024505,0.014535,0.100598,131472,30331,0.891103,0.992711,0,1,0.004444,0.006968,True
